# 🔬 Krypto: Mobile Forensic Activity Reconstruction Fine-Tuning
### LoRA Fine-Tuning on H100/A100 GPU with Native Hugging Face Trainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmanuelbadmus/Krypto/blob/main/FineTune_Gemma_Forensics.ipynb)

---
### 🎯 Objectives
1. **100% Citation Discipline**: Enforce strict `[EVT-xxxxxxxxxxxx]` ground-truth citations to eliminate hallucinations.
2. **Absence & SQLCipher Auditing**: Teach the model to explicitly document unrecoverable/encrypted data (Signal, Google Podcasts, WeChat residue).
3. **Multi-Signal Indirect Commute Deduction**: Synthesize unlogged Android Auto commutes from Bluetooth and charging logs.
4. **Beat Non-AI Baseline**: Outperform the 68.2% baseline recall in `baseline_benchmark_report.txt`.

## 🧹 Step 0: GPU Purge & Process Terminator (Run if GPU memory is clogged)

In [ ]:
import os
import gc
import torch

# 1. Kill any zombie/orphaned GPU processes
!fuser -k -9 /dev/nvidia* 2>/dev/null || true

# 2. Force PyTorch memory purge
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("=== CURRENT GPU MEMORY STATUS ===")
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv

## 🛠️ Step 1: Install Dependencies & Authenticate Hugging Face

In [ ]:
!pip install --upgrade --quiet torch transformers datasets peft accelerate bitsandbytes sentencepiece protobuf scikit-learn tabulate huggingface_hub

import os
import torch
from huggingface_hub import login

# Interactive Hugging Face login prompt
print("Please paste your Hugging Face Access Token when prompted below:")
login()

print(f"\nCUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")

## 📂 Step 2: Clone Public Repository & Setup Dataset Splits

In [ ]:
import os

# Clone public repository if running in Colab
if not os.path.exists("data"):
    !git clone https://github.com/emmanuelbadmus/Krypto.git
    %cd Krypto

train_file = "data/splits/train.jsonl"
val_file = "data/splits/val.jsonl"
gt_file = "data/raw_database/ground_truth.csv"
events_db_file = "data/raw_database/events_db.jsonl"

print("=" * 50)
print("  DATASET LOAD STATUS")
print("=" * 50)
print(f"  Working Directory: {os.getcwd()}")
print(f"  Train File: {train_file} -> Exists: {os.path.exists(train_file)}")
print(f"  Val File:   {val_file}   -> Exists: {os.path.exists(val_file)}")

with open(train_file) as f:
    print(f"  -> Loaded {len(f.readlines())} training windows (Date-Held-Out)")
with open(val_file) as f:
    print(f"  -> Loaded {len(f.readlines())} validation windows")

## 🧠 Step 3: Load Base Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b-it"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32,
    trust_remote_code=True,
)
model = model.to(device)
print(f"Base model loaded on {device}!")

## 💉 Step 4: Inject LoRA Target Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=32,                # LoRA Rank
    lora_alpha=64,       # LoRA Alpha scaling factor
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, peft_config, low_cpu_mem_usage=False)
model.print_trainable_parameters()

## 📊 Step 5: Format and Tokenize Datasets

In [ ]:
import gc
import json
import torch
from datasets import Dataset

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MAX_SEQ_LENGTH = 2048  # Perfectly fits all forensic events without OOM

def prepare_and_tokenize(filepath, tokenizer, max_len=MAX_SEQ_LENGTH):
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            msgs = item.get("messages", [])
            if len(msgs) >= 2:
                gemma_msgs = []
                system_text = ""
                for m in msgs:
                    if m["role"] == "system":
                        system_text = m["content"].strip() + "\n\n"
                    elif m["role"] == "user":
                        gemma_msgs.append({
                            "role": "user",
                            "content": (system_text + m["content"]).strip()
                        })
                        system_text = ""
                    elif m["role"] in ("assistant", "model"):
                        gemma_msgs.append({
                            "role": "assistant",
                            "content": m["content"]
                        })
                
                text = tokenizer.apply_chat_template(gemma_msgs, tokenize=False, add_generation_prompt=False)
                samples.append({"text": text})
    
    raw_ds = Dataset.from_list(samples)
    def tok_fn(batch):
        encoded = tokenizer(batch["text"], truncation=True, max_length=max_len)
        encoded["labels"] = encoded["input_ids"].copy()
        return encoded
        
    tokenized_ds = raw_ds.map(tok_fn, batched=True, remove_columns=["text"])
    return tokenized_ds

print("Tokenizing datasets for training...")
train_dataset = prepare_and_tokenize(train_file, tokenizer)
val_dataset = prepare_and_tokenize(val_file, tokenizer)

print(f"Ready: {len(train_dataset)} training samples | {len(val_dataset)} validation samples.")

## 🚀 Step 6: Train Model with Native Hugging Face Trainer (Memory Optimized)

In [ ]:
import gc
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Clear leftover CUDA cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

OUTPUT_DIR = "outputs_forensic_gemma_2b"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # Effective batch size = 8
    gradient_checkpointing=True,      # Saves 85% activation VRAM
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    logging_steps=1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Starting GPU Fine-Tuning...")
trainer_stats = trainer.train()
print(f"Training Complete! Runtime: {trainer_stats.metrics['train_runtime']:.2f}s")

## 💾 Step 7: Save Fine-Tuned LoRA Adapter

In [ ]:
final_adapter_dir = f"{OUTPUT_DIR}/final_adapter"
print(f"Saving LoRA adapter to {final_adapter_dir}...")
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print("Saved adapter successfully!")

## 🏆 Step 8: Automated In-Notebook Benchmark Evaluation
*(Scores Citation Precision, Absence Reasoning, and Hallucinations)*

In [ ]:
import re
from tabulate import tabulate

model.eval()
model.to(device)

print(f"Evaluating fine-tuned model on {val_file}...")
total_cited = 0
valid_cited = 0
hallucinated_cited = 0
absence_detected_count = 0
results = []

with open(val_file, "r", encoding="utf-8") as f:
    val_lines = [json.loads(l) for l in f if l.strip()]

for idx, item in enumerate(val_lines):
    msgs = item.get("messages", [])
    sys_prompt = msgs[0]["content"] if len(msgs) > 0 and msgs[0]["role"] == "system" else ""
    user_prompt = msgs[1]["content"] if len(msgs) > 1 and msgs[1]["role"] == "user" else ""
    target_answer = msgs[2]["content"] if len(msgs) > 2 and msgs[2]["role"] == "assistant" else ""

    valid_prompt_eids = set(re.findall(r"EVT-([a-f0-9]+)", user_prompt, flags=re.IGNORECASE))

    # Generate inference
    inp_msgs = [{"role": "user", "content": (sys_prompt + "\n\n" + user_prompt).strip()}]
    inputs = tokenizer.apply_chat_template(inp_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(inputs, max_new_tokens=300, temperature=0.1, use_cache=True)
    pred_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    pred_eids = re.findall(r"EVT-([a-f0-9]+)", pred_text, flags=re.IGNORECASE)
    v_c = [e for e in pred_eids if e in valid_prompt_eids]
    h_c = [e for e in pred_eids if e not in valid_prompt_eids]

    total_cited += len(pred_eids)
    valid_cited += len(v_c)
    hallucinated_cited += len(h_c)

    has_absence = any(kw in pred_text.lower() for kw in ["without direct artifact support", "sqlcipher", "encrypted", "unrecoverable"])
    if has_absence:
        absence_detected_count += 1

    prec = (len(v_c) / len(pred_eids)) if pred_eids else 1.0
    results.append([f"Window #{idx+1}", len(pred_eids), len(v_c), f"{prec*100:.1f}%", "✅" if has_absence else "-"])

precision = (valid_cited / total_cited * 100) if total_cited > 0 else 100.0
print("\n" + "=" * 65)
print("  FINAL FORENSIC EVALUATION SCORECARD")
print("=" * 65)
print(f"  Total Windows Evaluated:     {len(val_lines)}")
print(f"  Overall Citation Precision:  {precision:.2f}%")
print(f"  Hallucinated Phantom IDs:    {hallucinated_cited}")
print(f"  Absence Auditing Accuracy:   {(absence_detected_count / len(val_lines) * 100):.1f}%")
print("=" * 65)
print(tabulate(results[:10], headers=["Window", "Total Cited", "Valid", "Precision", "Absence"], tablefmt="grid"))

## 🔍 Step 9: Live Interactive Reconstruction Test

In [ ]:
# Test sample prompt
test_prompt = """Date: 2019-03-15
EXTRACTED ARTIFACTS:
[EVT-a9668b712bee] 17:33 Twitter tweet_seen_or_posted <224423919>
[EVT-b86eea5bdfcc] 08:03 Twitter tweet_seen_or_posted <804341497441255424>
PROVENANCE:
EVT-a9668b712bee -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 135
EVT-b86eea5bdfcc -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 488

Reconstruct the user activity for this window."""

sys_msg = "You are a digital forensic analyst. Reconstruct the chronological user activity from the extracted Android artifacts. Cite evidence using exact [EVT-xxxx] IDs. Explicitly state unrecoverable apps."
inp_msgs = [{"role": "user", "content": f"{sys_msg}\n\n{test_prompt}"}]

inputs = tokenizer.apply_chat_template(inp_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(inputs, max_new_tokens=300, temperature=0.1, use_cache=True)

print("=== RECONSTRUCTED OUTPUT ===")
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))